In [1]:
import os
import duckdb

con = duckdb.connect()                     # in-memory; tables vanish when the kernel restarts
con.sql("INSTALL httpfs; LOAD httpfs;")    # lets DuckDB read s3:// and https:// paths
con.sql("SET s3_region = 'us-west-2';")    # Overture's public bucket region

In [12]:
con.sql(r"""
DESCRIBE
SELECT 
    *,
FROM read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
""")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ gers_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fsq_place_id        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ address_a           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ address_b           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ locality_a          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ locality_b          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ region_a            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ region_b            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode_a          │ VARCHAR     │ YES     │ NULL    │ NULL  

In [21]:
# con.sql(r"""
# SELECT 
#     *,
#     (regexp_extract(address_a, '^\d+') <> regexp_extract(address_b, '^\d+')) AS number_mismatch,
#     (regexp_extract(address_a, '^\d+') = '' OR regexp_extract(address_b, '^\d+') = '') AS number_missing
# FROM read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
# WHERE address_b IS NOT NULL
# ORDER BY random()
# LIMIT 20
# """)

# con.sql(r"""
# SELECT 
#     COUNT(*) FILTER(
#         ((regexp_extract(address_a_clean, '^\d+') <> regexp_extract(address_b_clean, '^\d+'))
#         AND regexp_extract(address_a_clean, '^\d+') <> '' 
#         AND regexp_extract(address_b_clean, '^\d+') <> '')) 
#     AS number_mismatch,
#     COUNT(*) FILTER(
#         ((regexp_extract(address_a_clean, '^\d+') = '' OR regexp_extract(address_b_clean, '^\d+') = '') AND  
#         regexp_extract(address_a_clean, '^\d+') <> regexp_extract(address_b_clean, '^\d+') AND
#         (address_a_clean <> '' AND address_b IS NOT NULL)
#         )) 
#     AS number_missing
# FROM 
#     read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
# WHERE 
#     address_b IS NOT NULL
# """).df()

con.sql(r"""
COPY
(SELECT 
    *,
    ((regexp_extract(address_a_clean, '^\d+') <> regexp_extract(address_b_clean, '^\d+'))
    AND regexp_extract(address_a_clean, '^\d+') <> '' 
    AND regexp_extract(address_b_clean, '^\d+') <> '')
    AS number_mismatch,

    ((regexp_extract(address_a_clean, '^\d+') = '' OR regexp_extract(address_b_clean, '^\d+') = '') AND  
    regexp_extract(address_a_clean, '^\d+') <> regexp_extract(address_b_clean, '^\d+')
    )
    AS number_missing
FROM 
    read_parquet('Overture/ct_positives_clean_2026-09-23.1.parquet')
WHERE 
    address_b IS NOT NULL AND address_a_clean <> ''
) TO 'Overture/ct_positives_flagged_2026-09-23.1.parquet' (FORMAT parquet)
""")